Este script se usó para generar la tabla con la que se hizo el proceso del mapping de los valores incorrectos de los distintos atributos

In [0]:
# paquetes y funciones
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.types import StringType


# funcion que hace explode de arrays
def explode_arrays(df):
    array_columns = []
    
    # busca y selecciona las columnas array
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # itera y explota las columnas array
    for col_name in array_columns:
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verifica si el tipo de dato del campo creado exploded_ es un struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # Si lo es, extrae los elementos o subcampos, crea 1 campo nuevo por cada elemento encontrado y elimina el campo original, o sea el array
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # si no es un StructType (ie, StringType o DoubleType), reemplaza el campo array original con la columna explotada
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # elimina la columna temporal exploded_
        df = df.drop(f"exploded_{col_name}")
    
    return df


# Función para separar el campo usando '~' y ',' como delimitadores
def split_multiple_delimiters(df, input_col, output_col):
    # Usamos regexp_replace para normalizar los delimitadores a uno solo (por ejemplo ',')
    normalized_col = F.regexp_replace(input_col, '[~,]+', ',')
    # Hacemos split del resultado normalizado
    return df.withColumn(output_col, F.split(normalized_col, ','))


# Definir la función para capitalizar la primera letra, limpiar espacios y reemplazar '_'
def clean_text(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.trim(F.regexp_replace(F.col(input_col), '_', ' '))
    )

# funcion para extraer el contenido entre corchetes
def extract_string_content(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.when(
            F.col(input_col).rlike(r'\[.*?\]'),  # Si contiene corchetes
            F.regexp_replace(  # Remover las comillas dobles después de extraer el contenido
                F.regexp_extract(F.col(input_col), r'\[(.*?)\]', 1),
                r'"', ''  # Reemplazar todas las comillas dobles por un string vacío
            )
        ).otherwise(F.col(input_col))  # Si no tiene corchetes, dejar el valor original
    )


In [0]:
# Import tables

################ dim_gdm table #################
dim_gdm = spark.table("crm_reporting.dim_gdm_brand_profile")
# print(dim_gdm.count()) #12769048
# print(len(gdm.columns))


####### FIRST SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE
df = dim_gdm
# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]
# completely_null_columns = [c for c in df.columns if non_null_counts[c] == 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 78
# print(len(completely_null_columns)) ## 117
# print(completely_null_columns)

# select the founded fields
gdm_cols_filt = df.select(non_completely_null_columns).\
  withColumnRenamed("created_dt",'created_date').\
  withColumn("created_date",F.to_date('created_date'))



##### OTHER COLUMNS TO REMOVE
columns_to_remove = ['dept_store_regional','etl_batch_id','mdm_source','sys_created_by','beauty_supply_store','category','product_category']

##### Filter columns that contain the string '_dt' and the ones in columns_to_remove
columns_wt_dt = [col for col in gdm_cols_filt.columns if '_dt' not in col and col not in columns_to_remove]

# count of not-enterely null columns
# print(len(columns_wt_dt)) ## 51
#print(non_completely_null_columns)


# select the list of columns to keep
gdm_cols_filt = gdm_cols_filt.select(columns_wt_dt)
gdm_cols_filt.createOrReplaceTempView("gdm_cols_filt_vw")

In [0]:
# call the explode array function twice
expld_gdm_1 = explode_arrays(gdm_cols_filt) # explode first array levels
expld_gdm_1 = explode_arrays(expld_gdm_1) # explode second array levels

# print(expld_gdm_1.count()) # 433098
# print(len(expld_gdm.columns)) ## 81
# display(expld_gdm.limit(1))
expld_gdm_1.createOrReplaceTempView("expld_gdm_1_vw")
# print(expld_gdm_1.count()) # 13155657

In [0]:
# SECOND SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE

df = expld_gdm_1

# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 47

# select the list of columns to keep
def_gdm_v1 = df.select(non_completely_null_columns).\
  drop('analysis_exact_age','analysis_calculated_age')

# limpiamos valores con la estructura: Concern_type="texture uniformity";zone=["whole face", "cheek"]
# def_gdm_v1 = extract_string_content(def_gdm_v1, "analysis_analysis_concern_zone", "analysis_analysis_concern_zone")
def_gdm_v1.createOrReplaceTempView("def_gdm_v1_vw")

In [0]:
###  UNPIVOT DEF_GDM
# 1. Señalamos las columnas que no son atributos
non_attributes = ["brand_code","brand_country",'created_date','brand_mdm_id']
attributes = [c for c in def_gdm_v1.columns if c not in non_attributes] 

# 2. Crear la expresión para el unpivot con stack()
num_atributos = len(attributes)
stack_expr = ", ".join(
    [f"'{col}', {col}" for col in attributes]
)

# 3. Realizar el unpivot usando `stack()`
gdm_unpivot = def_gdm_v1.select(
    *non_attributes,
    F.expr(f"stack({num_atributos}, {stack_expr}) as (attribute, value)")).distinct()

# remove white spaces and capitalize values
# gdm_unpivot = clean_text(gdm_unpivot, "value", "value_k")

#remover lineas nulas
# gdm_unpivot = gdm_unpivot.filter(F.col("value").isNotNull())

# print(gdm_unpivot.count()) # 36169590 lineas quitando nulos, 12457382 distinct brand_mdm_id
gdm_unpivot.createOrReplaceTempView("gdm_unpivot_vw")

# tmp = gdm_unpivot.select('value','value_k').distinct()
# display(tmp)

In [0]:
## IDENTIFICA IDS CON ATRIBUTOS HASHEADOS
# Filtrar valores que tienen 6 o menos caracteres numericos para quitar hasheados
hashed = gdm_unpivot.filter(F.col("value").rlike(r'(.*[0-9]){6,}')).\
  select('brand_mdm_id').distinct()  #65047 con hasheados
hashed.createOrReplaceTempView("hashed_vw")

query = f"""
select
  a.* 
from gdm_unpivot_vw a
inner join hashed_vw b
  on a.brand_mdm_id = b.brand_mdm_id 
"""

hashed_all_attrib = spark.sql(query)
display(hashed_all_attrib)



# tmp = gdm_unpivot.select('brand_mdm_id').distinct() # 12635107 total

# print(tmp.count())

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

In [0]:
## SE AGRUPAN LOS IDS POR ATRIBUTO Y VALOR
gdm_unpivot_gp = gdm_unpivot.groupBy("attribute",'value').agg(
        F.countDistinct("brand_mdm_id").alias("id_counts"),
        #F.min("created_date").alias("min_date"),
        F.max("created_date").alias("max_date")
    )

# print(gdm_unpivot_gp.count())
gdm_unpivot_gp.createOrReplaceTempView("gdm_unpivot_gp_vw")
# display(tmp)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
data = [
    ('analysis_analysis_clinical_sign_clinical_sign','CLINICAL_SIGN'),
    ('analysis_analysis_clinical_sign_sign_type','SIGN_TYPE'),
    ('analysis_analysis_clinical_sign_zone','ZONE'),
    ('analysis_analysis_concern_concern','CONCERN'),
    ('analysis_analysis_concern_concern_type','CONCERN_TYPE'), 
    ('analysis_analysis_concern_zone','ZONE'),
    ('analysis_analysis_type','ANALYSIS_TYPE'),
    ('channel_preference_product_purchase_channel','PRODUCT_PURCHASE_CHANNEL'),
    ('concern_improvement_goal_concern','CONCERN'),
    ('concern_zone','ZONE'),
    ('desired_color_finish','HAIR_COLOR_FINISH'),
    ('desired_color_permanence','HAIR_COLOR_PERMANENCE'), 
    ('desired_cover','HAIR_COLOR_COVER'), 
    ('fragrance_routine_perfume_moment','PERFUME_MOMENT'),
    ('fragrance_priority','FRAGRANCE_MOTIVATION'),
    ('hair_care_product_used','HAIRCARE_PRODUCT'),
    ('hair_routine_heating_heating_tool','HEATING_TOOL'),
    ('improvement_goal_concern','CONCERN'),
    ('last_hair_color_service','HAIR_COLOR_SERVICE'),
    ('last_hair_style_look','HAIR_STYLE_LOOK'),
    ('last_skincare_medical_treatment','LAST_SKINCARE_PROFESSIONAL_TREATMENT'),
    ('left_eye_color','EYE_COLOR'),
    ('look_occasion_makeup','MAKEUP_LOOK_OCCASION'),
    ('makeup_priority','MAKEUP_EXPECTATION'),
    ('natural_hair_color','HAIR_COLOR'),
    ('right_eye_color','EYE_COLOR'),
    ('skin_sensitivity_skin_sensitivity','SKIN_SENSITIVITY'),
    ('skin_sensitivity_zone','ZONE'),
    ('skin_type_skin_type','SKIN_TYPE'),
    ('desired_color_finish','ZONE'), 
    ('desired_style','HAIR_STYLE'), 
    ('skin_type_zone','ZONE')
]



# Crear el DataFrame
columns = ["attribute", "attribute_k"]
attributes_mapping = spark.createDataFrame(data, columns)
attributes_mapping.createOrReplaceTempView("attributes_mapping_vw")


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
bmdm_lkp = spark.table("prod_latam_catalog.crm_reporting.lkp_bmdm_reference")
# bmdm_global = spark.table("prod_latam_catalog.crm_analytics.bmdm_mapping_lkp_table")

# some transformations
bmdm_lkp = bmdm_lkp.withColumn('reference_name', F.lower('reference_name')).\
  select('reference_name','reference_value').\
  orderBy('reference_name','reference_value')	
# bmdm_lkp = clean_text(bmdm_lkp, "reference_value", "value_k")

bmdm_lkp.createOrReplaceTempView("bmdm_lkp_vw")
# display(bmdm_lkp.limit(5))

# tmp = bmdm_lkp.filter(F.col("reference_name").isin(['ROUTINE_GOAL','ANALYSIS_TYPE'])).\
#   select('reference_name','value_k','reference_value').distinct()
display(bmdm_lkp)



com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
## CRUCE CON LA LKP
##  IMPORTANTE! dim_gdm_mapping es la tabla FINAL que contiene los valores originales incorrectos a mapear, de esta tabla se inició el proceso del mapping
dim_gdm_mapping = spark.sql("""
with merge_1 as (
select
  a.*,
  CASE WHEN b.attribute_k IS NULL THEN a.attribute
    ELSE lower(b.attribute_k) END as attribute_k
from gdm_unpivot_gp_vw a
left join attributes_mapping_vw b
  on a.attribute = b.attribute
)

--merge_2 as (
select 
  a.*,  
  c.reference_value as value_lkp
from merge_1 a
left join bmdm_lkp_vw c
  on a.attribute_k = c.reference_name
  and a.value = c.reference_value
--),
""")


# Filtrar valores que tienen 6 o menos caracteres numericos para quitar hasheados
dim_gdm_mapping = dim_gdm_mapping.filter(~F.col("value").rlike(r'(.*[0-9]){6,}'))

#display(tmp)
# print(dim_gdm_mapping.count()) # 33024 opciones sin split
dim_gdm_mapping.createOrReplaceTempView("dim_gdm_mapping_vw")


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage